# Single-image captioning — beam search inference

Give it **one image** and **one checkpoint path**, and it generates a caption with **beam search**. Use it to demo a trained model on your own pictures.

- The checkpoint stores its own `config`, so the right architecture (CNN/ViT/CLIP + GPT-2, or CNN+GRU) is rebuilt automatically.
- No dataset download is needed for GPT-2 models. (A CNN+GRU checkpoint needs the word vocabulary, so the notebook fetches just the COCO *annotations* in that one case.)
- `beam_size=1` is greedy; higher widths search more. Built for Colab (also runs locally).

## 1. Install dependencies and imports

In [ ]:
# Install once if needed
!pip -q install transformers nltk

import os, json, random, math, textwrap
from dataclasses import dataclass, fields
from typing import Optional
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision.transforms as T
from torchvision import models

from transformers import (
    AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer,
    AutoImageProcessor, CLIPVisionModel,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

## 2. Settings & mount Drive

- `MODEL_PATH` — the checkpoint to run.
- `IMAGE_PATH` — the image to caption. Leave it `""` to be prompted to **upload** a file (Colab) instead.
- `BEAM_SIZE` — beam width for the single caption. `SHOW_BEAM_COMPARISON` also prints the caption at each width in `COMPARE_BEAMS`.

In [ ]:
# Portable paths: works on Colab (mounts Drive) AND a local Jupyter notebook.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content"
    ON_COLAB = True
except ImportError:
    BASE_DIR = os.path.abspath(".")
    ON_COLAB = False

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")   # only used for GRU vocab
os.makedirs(DATA_DIR, exist_ok=True)

# ---- Inputs (edit these) ----------------------------------------------------
MODEL_PATH = "/content/drive/MyDrive/image_captioning_finetune/vit_gpt2_best.pt"
IMAGE_PATH = ""           # path to your image; "" -> upload prompt on Colab

# Ground-truth caption(s) for THIS image, needed to compute BLEU-4.
# Leave empty to skip scoring. For a COCO image, paste its human captions here.
REFERENCE_CAPTIONS = [
    # "a man riding a skateboard down a street",
    # "a person on a skateboard on the road",
]

# ---- Beam search knobs ------------------------------------------------------
BEAM_SIZE = 3             # 1 = greedy
LENGTH_PENALTY = 1.0      # >1 favours longer captions, <1 shorter
MAX_GEN_LEN = 40
MAX_TEXT_LEN = 40

SHOW_BEAM_COMPARISON = True
COMPARE_BEAMS = [1, 2, 3, 5]

# ---- Nucleus (top-p) sampling knobs -----------------------------------------
TOP_P = 0.9               # nucleus mass kept; smaller = more conservative
TEMPERATURE = 1.0         # >1 flatter/more random, <1 sharper/greedier
NUM_NUCLEUS_SAMPLES = 3   # how many sampled captions to draw (it's stochastic)

COCO_SPLIT = "val"        # only relevant if the checkpoint is a CNN+GRU model

print("Model path:", MODEL_PATH)
print("Image path:", IMAGE_PATH or "(will prompt to upload)")
print("Beam size:", BEAM_SIZE)
print(f"Nucleus: top_p={TOP_P}, temperature={TEMPERATURE}, samples={NUM_NUCLEUS_SAMPLES}")

## 3. Encoder/decoder framework

Same building blocks the training notebooks used, so a checkpoint's saved `config` rebuilds the identical model. `rnn_vocab` is left empty here and only built later if a GRU checkpoint needs it.

In [ ]:
# ---- Word-level vocabulary (only used for GRU checkpoints) -------------------
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = Counter()

    def build(self, captions):
        for cap in captions:
            self.word_count.update(word_tokenize(cap.lower()))
        idx = 4
        for w, c in self.word_count.items():
            if c >= self.freq_threshold:
                self.word2idx[w] = idx; self.idx2word[idx] = w; idx += 1

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), "<unk>")
            if w == "<end>": break
            if w not in ("<start>", "<pad>"): words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

rnn_vocab = None   # built on demand for GRU checkpoints

# ---- Image preprocessing ----------------------------------------------------
resnet_transform = T.Compose([
    T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def preprocess_images(images, encoder_kind, image_processor):
    if encoder_kind == "cnn":
        return torch.stack([resnet_transform(im) for im in images], 0)
    return image_processor(list(images), return_tensors="pt").pixel_values

# ---- Encoder: image -> sequence of feature vectors (B, S, D_enc) ------------
class ImageEncoder(nn.Module):
    def __init__(self, kind, name):
        super().__init__()
        self.kind = kind
        if kind == "cnn":
            resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.backbone = nn.Sequential(*list(resnet.children())[:-2])
            self.feat_dim = 2048
        elif kind == "clip":
            self.backbone = CLIPVisionModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size
        else:  # vit
            self.backbone = AutoModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size

    def forward(self, images):
        if self.kind == "cnn":
            f = self.backbone(images)
            B, C, H, W = f.shape
            return f.view(B, C, H * W).permute(0, 2, 1)
        out = self.backbone(pixel_values=images)
        return out.last_hidden_state

# ---- Decoder A: word-level GRU ----------------------------------------------
class GRUDecoder(nn.Module):
    def __init__(self, feat_dim, vocab, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.vocab = vocab
        self.pad_id = vocab.word2idx["<pad>"]
        self.end_id = vocab.word2idx["<end>"]
        self.img_proj = nn.Linear(feat_dim, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        self.embed = nn.Embedding(len(vocab), embed_size)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, len(vocab))

    def _img_token(self, enc_seq):
        pooled = enc_seq.mean(dim=1)
        return self.bn(self.img_proj(pooled))

    def decode(self, ids):
        return self.vocab.decode(ids)

# ---- Decoder B: GPT-2 with cross-attention ----------------------------------
class GPT2Decoder(nn.Module):
    def __init__(self, feat_dim, tokenizer, dropout=0.1, freeze_base=False):
        super().__init__()
        cfg = AutoConfig.from_pretrained("gpt2")
        cfg.is_decoder = True
        cfg.add_cross_attention = True
        cfg.resid_pdrop = dropout
        cfg.embd_pdrop = dropout
        cfg.attn_pdrop = dropout
        self.gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", config=cfg)
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.enc_proj = nn.Linear(feat_dim, cfg.n_embd)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()

# ---- Full model -------------------------------------------------------------
class CaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

    @torch.no_grad()
    def encode(self, images):
        return self.encoder(images)

    def decode(self, ids):
        return self.decoder.decode(ids)

# ---- Config + builder -------------------------------------------------------
@dataclass
class ExperimentConfig:
    name: str
    encoder_kind: str
    encoder_name: str
    decoder: str
    learning_rate: float
    weight_decay: float = 0.0
    dropout: float = 0.1
    freeze_gpt2_base: bool = False
    embed_size: int = 256
    hidden_size: int = 512
    num_layers: int = 2
    freq_threshold: int = 5
    batch_size: int = 16
    epochs: int = 15
    max_train_batches: Optional[int] = 300

def build_model(config):
    encoder = ImageEncoder(config.encoder_kind, config.encoder_name)
    feat_dim = encoder.feat_dim
    if config.decoder == "gru":
        assert rnn_vocab is not None, "GRU checkpoint needs rnn_vocab (built in the load cell)."
        decoder = GRUDecoder(feat_dim, rnn_vocab, config.embed_size,
                             config.hidden_size, config.num_layers, config.dropout)
    elif config.decoder == "gpt2":
        tok = AutoTokenizer.from_pretrained("gpt2")
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<|pad|>"})
        decoder = GPT2Decoder(feat_dim, tok, dropout=config.dropout,
                              freeze_base=config.freeze_gpt2_base)
    else:
        raise ValueError(f"Unknown decoder: {config.decoder}")
    model = CaptioningModel(encoder, decoder, freeze_encoder=True)
    image_processor = (None if config.encoder_kind == "cnn"
                       else AutoImageProcessor.from_pretrained(config.encoder_name))
    return {"model": model, "encoder_kind": config.encoder_kind,
            "decoder_kind": config.decoder, "image_processor": image_processor}

print("Framework ready.")

## 4. Load the checkpoint

Rebuilds the architecture from the stored `config` (inferring any fields older checkpoints omit), then loads the weights. If it's a CNN+GRU checkpoint, the COCO annotations are downloaded once to rebuild the word vocabulary.

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=device)

# Infer/fill fields that some checkpoints (e.g. the ViT-only notebook) don't store.
saved_cfg = dict(ckpt["config"])
if "encoder_kind" not in saved_cfg:
    _nm = str(saved_cfg.get("encoder_name", "")).lower()
    if "clip" in _nm:
        saved_cfg["encoder_kind"] = "clip"
    elif "resnet" in _nm or _nm == "cnn":
        saved_cfg["encoder_kind"] = "cnn"
    else:
        saved_cfg["encoder_kind"] = "vit"
saved_cfg.setdefault("name", os.path.splitext(os.path.basename(MODEL_PATH))[0])
saved_cfg.setdefault("decoder", "gpt2")

valid = {f.name for f in fields(ExperimentConfig)}
config = ExperimentConfig(**{k: v for k, v in saved_cfg.items() if k in valid})
MODEL_NAME = config.name

# A GRU checkpoint needs the word vocabulary -> fetch annotations and build it.
if config.decoder == "gru":
    import urllib.request, zipfile
    ann_dir = os.path.join(DATA_DIR, "annotations")
    ann_file = os.path.join(ann_dir, f"captions_{COCO_SPLIT}2017.json")
    if not os.path.exists(ann_file):
        print("Downloading COCO annotations (needed for GRU vocab)...")
        zp = os.path.join(DATA_DIR, "annotations.zip")
        urllib.request.urlretrieve(
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(DATA_DIR)
        os.remove(zp)
    with open(ann_file, "r") as f:
        _coco = json.load(f)
    _ids = list({img["id"] for img in _coco["images"]})
    random.Random(42).shuffle(_ids)
    _train_ids = set(_ids[:int(0.90 * len(_ids))])   # same 90/10 split as training
    _train_caps = [a["caption"] for a in _coco["annotations"] if a["image_id"] in _train_ids]
    rnn_vocab = Vocabulary(freq_threshold=config.freq_threshold)
    rnn_vocab.build(_train_caps)
    print("Built GRU vocab:", len(rnn_vocab), "words")

bundle = build_model(config)
model = bundle["model"].to(device)
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval()

print("Loaded:", MODEL_NAME)
print(f"  encoder = {config.encoder_kind} ({config.encoder_name})")
print(f"  decoder = {config.decoder}")
print(f"  reported val BLEU-4 at save time = {ckpt.get('bleu4')} (epoch {ckpt.get('epoch')})")
if missing:    print("  [warn] missing keys:", len(missing))
if unexpected: print("  [warn] unexpected keys:", len(unexpected))

## 5. Beam search decoders

`beam_size` hypotheses ranked by cumulative log-probability; finished hypotheses scored with length normalisation. **`beam_size=1` == greedy.** GPT-2 re-runs over the growing sequence each step feeding `encoder_hidden_states`; GRU carries a per-beam hidden state.

In [ ]:
@torch.no_grad()
def beam_search_gpt2(decoder, enc_seq_1, max_len, beam_size, length_penalty=1.0):
    dev = enc_seq_1.device
    enc_hidden = decoder.enc_proj(enc_seq_1)
    bos, eos = decoder.bos_id, decoder.eos_id
    beams = [([bos], 0.0)]
    finished = []
    for _ in range(max_len):
        if not beams:
            break
        input_ids = torch.tensor([b[0] for b in beams], device=dev)
        nb = input_ids.size(0)
        enc_b = enc_hidden.expand(nb, -1, -1).contiguous()
        out = decoder.gpt2(input_ids=input_ids, encoder_hidden_states=enc_b)
        logprobs = F.log_softmax(out.logits[:, -1, :], dim=-1)
        V = logprobs.size(-1)
        total = torch.tensor([b[1] for b in beams], device=dev).unsqueeze(1) + logprobs
        cand = min(2 * beam_size, total.numel())
        top_scores, top_flat = total.view(-1).topk(cand)
        new_beams = []
        for s, fi in zip(top_scores.tolist(), top_flat.tolist()):
            b, t = divmod(int(fi), V)
            seq = beams[b][0] + [t]
            if t == eos:
                finished.append((seq, s / (max(len(seq) - 1, 1) ** length_penalty)))
            else:
                new_beams.append((seq, s))
            if len(new_beams) == beam_size:
                break
        beams = new_beams
        if len(finished) >= beam_size:
            break
    if finished:
        best = max(finished, key=lambda x: x[1])[0]
    else:
        best = max(beams, key=lambda b: b[1] / (max(len(b[0]) - 1, 1) ** length_penalty))[0]
    return [t for t in best if t not in (bos, eos)]


@torch.no_grad()
def beam_search_gru(decoder, enc_seq_1, max_len, beam_size, length_penalty=1.0):
    dev = enc_seq_1.device
    end = decoder.end_id
    feat = decoder._img_token(enc_seq_1)
    out, states = decoder.rnn(feat.unsqueeze(1))
    logprobs = F.log_softmax(decoder.linear(out.squeeze(1)), dim=-1)[0]
    topv, topi = logprobs.topk(beam_size)
    seqs = [[int(t)] for t in topi.tolist()]
    scores = topv.clone()
    states = states.repeat(1, beam_size, 1)
    finished = []
    for _ in range(max_len - 1):
        if not seqs:
            break
        last = torch.tensor([s[-1] for s in seqs], device=dev)
        emb = decoder.embed(last).unsqueeze(1)
        out, states = decoder.rnn(emb, states)
        lp = F.log_softmax(decoder.linear(out.squeeze(1)), dim=-1)
        V = lp.size(-1)
        total = scores.unsqueeze(1) + lp
        cand = min(2 * beam_size, total.numel())
        top_scores, top_flat = total.view(-1).topk(cand)
        new_seqs, new_scores, parent = [], [], []
        for s, fi in zip(top_scores.tolist(), top_flat.tolist()):
            b, t = divmod(int(fi), V)
            if t == end:
                finished.append((seqs[b], s / (max(len(seqs[b]), 1) ** length_penalty)))
            else:
                new_seqs.append(seqs[b] + [t]); new_scores.append(s); parent.append(b)
            if len(new_seqs) == beam_size:
                break
        if not new_seqs:
            break
        seqs = new_seqs
        scores = torch.tensor(new_scores, device=dev)
        states = states[:, torch.tensor(parent, device=dev), :].contiguous()
        if len(finished) >= beam_size:
            break
    if finished:
        return max(finished, key=lambda x: x[1])[0]
    return max(zip(seqs, scores.tolist()),
               key=lambda z: z[1] / (max(len(z[0]), 1) ** length_penalty))[0]


@torch.no_grad()
def caption_beam(model, pixel_values_1, max_len, beam_size, length_penalty=1.0):
    enc_seq = model.encode(pixel_values_1)
    dec = model.decoder
    if isinstance(dec, GPT2Decoder):
        ids = beam_search_gpt2(dec, enc_seq, max_len, beam_size, length_penalty)
    else:
        ids = beam_search_gru(dec, enc_seq, max_len, beam_size, length_penalty)
    return model.decode(ids)

print("Beam search ready.")

## 5b. Nucleus (top-p) sampling decoders

Instead of always taking the highest-probability token (greedy/beam), **sample** from
the model's distribution restricted to the smallest set of tokens whose cumulative
probability reaches `top_p`. This breaks the repetition attractor *without* any explicit
anti-repetition rule. It is **stochastic**, so each call can differ — draw several.

- `top_p` smaller -> safer/more modal; larger -> more diverse.
- `temperature` >1 flattens the distribution (more random), <1 sharpens it.


In [ ]:
@torch.no_grad()
def _nucleus_pick(logits, top_p, temperature):
    """Sample one token id from the top-p (nucleus) of a 1-D logits vector."""
    logits = logits / max(temperature, 1e-6)
    probs = F.softmax(logits, dim=-1)
    sp, si = torch.sort(probs, descending=True)
    cdf = torch.cumsum(sp, dim=-1)
    keep = cdf <= top_p
    keep[0] = True                       # always keep the most likely token
    sp, si = sp[keep], si[keep]
    nxt = si[torch.multinomial(sp / sp.sum(), 1)]
    return int(nxt.item())


@torch.no_grad()
def nucleus_sample_gpt2(decoder, enc_seq_1, max_len, top_p=0.9, temperature=1.0):
    dev = enc_seq_1.device
    enc_hidden = decoder.enc_proj(enc_seq_1)
    bos, eos = decoder.bos_id, decoder.eos_id
    seq = [bos]
    for _ in range(max_len):
        ids = torch.tensor([seq], device=dev)
        out = decoder.gpt2(input_ids=ids, encoder_hidden_states=enc_hidden)
        nxt = _nucleus_pick(out.logits[:, -1, :].squeeze(0), top_p, temperature)
        if nxt == eos:
            break
        seq.append(nxt)
    return [t for t in seq if t not in (bos, eos)]


@torch.no_grad()
def nucleus_sample_gru(decoder, enc_seq_1, max_len, top_p=0.9, temperature=1.0):
    dev = enc_seq_1.device
    end = decoder.end_id
    feat = decoder._img_token(enc_seq_1)
    out, states = decoder.rnn(feat.unsqueeze(1))         # image -> first token
    nxt = _nucleus_pick(decoder.linear(out.squeeze(1)).squeeze(0), top_p, temperature)
    seq = []
    for _ in range(max_len):
        if nxt == end:
            break
        seq.append(nxt)
        emb = decoder.embed(torch.tensor([nxt], device=dev)).unsqueeze(1)
        out, states = decoder.rnn(emb, states)
        nxt = _nucleus_pick(decoder.linear(out.squeeze(1)).squeeze(0), top_p, temperature)
    return seq


@torch.no_grad()
def caption_nucleus(model, pixel_values_1, max_len, top_p=0.9, temperature=1.0):
    enc_seq = model.encode(pixel_values_1)
    dec = model.decoder
    if isinstance(dec, GPT2Decoder):
        ids = nucleus_sample_gpt2(dec, enc_seq, max_len, top_p, temperature)
    else:
        ids = nucleus_sample_gru(dec, enc_seq, max_len, top_p, temperature)
    return model.decode(ids)

print("Nucleus sampling ready.")


## 6. Caption the image

Loads the image (or prompts for an upload), runs beam search, and shows the picture with its caption.

In [ ]:
# Load the image.
if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    image = Image.open(IMAGE_PATH).convert("RGB")
    src = IMAGE_PATH
elif ON_COLAB:
    from google.colab import files
    print("Choose an image to upload:")
    up = files.upload()
    src = next(iter(up))
    image = Image.open(src).convert("RGB")
else:
    raise FileNotFoundError("Set IMAGE_PATH to a valid image file (no upload prompt off Colab).")

# Preprocess once for this model's encoder, then caption with the chosen beam size.
pixel_values = preprocess_images([image], bundle["encoder_kind"],
                                 bundle["image_processor"]).to(device)
caption = caption_beam(model, pixel_values, MAX_GEN_LEN, BEAM_SIZE, LENGTH_PENALTY)

plt.figure(figsize=(7, 7))
plt.imshow(image); plt.axis("off")
plt.title(textwrap.fill(f"[{MODEL_NAME} | beam {BEAM_SIZE}]  {caption}", 50), fontsize=11)
plt.tight_layout(); plt.show()

print("Source :", src)
print("Caption:", caption)

## 7. (Optional) Caption at several beam sizes

See how the caption changes as the beam widens — useful for spotting greedy repetition vs the beam-search length bias.

In [ ]:
if SHOW_BEAM_COMPARISON:
    print(f"Captions for {MODEL_NAME}:\n")
    for bs in COMPARE_BEAMS:
        cap = caption_beam(model, pixel_values, MAX_GEN_LEN, bs, LENGTH_PENALTY)
        tag = " (greedy)" if bs == 1 else ""
        print(f"  beam {bs}{tag}: {cap}")
else:
    print("SHOW_BEAM_COMPARISON is False - skipping.")

## 8. Compare decoding strategies on this image

Greedy and beam are deterministic (one caption each). Nucleus is stochastic, so we draw
`NUM_NUCLEUS_SAMPLES` captions — notice they vary, and tend to avoid the greedy/beam
repetition loop. (BLEU usually favours greedy/beam since references are modal text;
nucleus trades a little overlap for fluency and diversity.)


In [ ]:
# Reuses `pixel_values` and `image` from the captioning cell above.
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
_smooth = SmoothingFunction().method1

def bleu4(caption, refs):
    """Sentence-level BLEU-4 of `caption` against a list of reference strings."""
    if not refs:
        return None
    hyp = word_tokenize(caption.lower())
    if not hyp:
        return 0.0
    refs_tok = [word_tokenize(r.lower()) for r in refs]
    return sentence_bleu(refs_tok, hyp, weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=_smooth)

def _fmt(score):
    return f"   (BLEU-4 {score:.4f})" if score is not None else ""

refs = [r for r in REFERENCE_CAPTIONS if r.strip()]
if not refs:
    print("(REFERENCE_CAPTIONS is empty -> BLEU-4 needs ground-truth captions; "
          "scores skipped.)\n")

print(f"Decoding comparison for {MODEL_NAME}:\n")

greedy_cap = caption_beam(model, pixel_values, MAX_GEN_LEN, 1, LENGTH_PENALTY)
g_bleu = bleu4(greedy_cap, refs)
print(f"  greedy            : {greedy_cap}{_fmt(g_bleu)}")

beam_cap = caption_beam(model, pixel_values, MAX_GEN_LEN, BEAM_SIZE, LENGTH_PENALTY)
b_bleu = bleu4(beam_cap, refs)
print(f"  beam (k={BEAM_SIZE})         : {beam_cap}{_fmt(b_bleu)}")

print(f"\n  nucleus (top_p={TOP_P}, T={TEMPERATURE}) -- {NUM_NUCLEUS_SAMPLES} draws:")
nucleus_caps, nuc_bleus = [], []
for i in range(NUM_NUCLEUS_SAMPLES):
    nc = caption_nucleus(model, pixel_values, MAX_GEN_LEN, TOP_P, TEMPERATURE)
    sc = bleu4(nc, refs)
    nucleus_caps.append(nc); nuc_bleus.append(sc)
    print(f"    sample {i + 1}: {nc}{_fmt(sc)}")
if refs:
    valid = [s for s in nuc_bleus if s is not None]
    print(f"    -> nucleus BLEU-4: mean {sum(valid) / len(valid):.4f}, "
          f"best {max(valid):.4f}")

# Show the image with one caption + score from each strategy.
def _lbl(score):
    return f"  [BLEU-4 {score:.3f}]" if score is not None else ""

lines = [
    f"greedy: {greedy_cap}{_lbl(g_bleu)}",
    f"beam{BEAM_SIZE}: {beam_cap}{_lbl(b_bleu)}",
    f"nucleus: {nucleus_caps[0]}{_lbl(nuc_bleus[0])}",
]
title = "\n".join(textwrap.fill(l, 60) for l in lines)

plt.figure(figsize=(7, 7))
plt.imshow(image); plt.axis("off")
plt.title(title, fontsize=10, loc="left")
plt.tight_layout(); plt.show()
